In [18]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if not (root / "src").exists():
    root = root.parent

if not (root / "data").exists():
    root = root.parent

if not (root / "outputs").exists():
    root = root.parent

if str(root) not in sys.path:
    sys.path.append(str(root))

from src.aggregate import *
from src.create_csv import *
from src.get_match import *
from src.scores import *
from src.load_data import *

In [19]:
#type checking
from src.validation import assert_column_types
mc = load_match_context()

expected_context_types = {
    "match_name": "string",
    "stage": "string",
    "favored_team": "string",
    "underdog_team": "string",
    "winner": "string",
    "spread": "numeric"
}
assert_column_types(mc, expected_context_types)

In [20]:
#READ ERROR MESSAGE IF APPEARS
create_all_processed()
cleaned = load_cleaned_data()
match_sheet = load_match_sheet()
team_sheet = load_team_sheet()
stage_sheet = load_stage_sheet()
incidents = load_incidents()

In [21]:
#data validation
assert team_sheet["matches_played"].notna().all()
assert match_sheet["incident_count"].sum() == len(incidents)
assert stage_sheet.notna().any().any()
assert incidents.notna().any().any()

The following code snippets are just sanity checks. Take a look and make sure values make sense, or else Notebook 3 will make poor analysis and graphs.

In [22]:
match_sheet.head(100)

,match_name,match_bias_score,abs_bias_significance,stage,incident_count,favored_team,underdog_team,winner,spread,upset,bias_direction,VAR_count
0,south africa vs canada,-1.02,1.02,R32,1,canada,south africa,canada,-1,False,underdog,0
1,brazil vs japan,-0.16,0.16,R32,1,brazil,japan,brazil,-1,False,neutral,0
2,germany vs paraguay,-2.30,2.30,R32,2,germany,paraguay,paraguay,0,True,underdog,1
3,mexico vs ecuador,0.10,0.10,R32,1,mexico,ecuador,mexico,-2,False,neutral,0
4,usa vs bosnia and herzegovina,0.22,1.65,R32,4,usa,bosnia and herzegovina,usa,-1,False,neutral,1
5,portugal vs croatia,0.30,0.30,R32,1,portugal,croatia,portugal,-1,False,neutral,1
6,switzerland vs algeria,0.88,0.88,R32,1,switzerland,algeria,switzerland,-1,False,favorite,0
7,australia vs egypt,0.33,0.33,R32,2,egypt,australia,egypt,0,False,neutral,0
8,argentina vs cape verde,3.26,3.26,R32,4,argentina,cape verde,argentina,-1,False,favorite,0
9,norway vs ivory coast,0.30,0.30,R32,1,norway,ivory coast,norway,-1,False,neutral,0


In [23]:
team_sheet.head(10)

,team,matches_played,incident_count,impact,impact_per_match
0,canada,2,4,-1.91,-0.96
1,brazil,2,3,1.06,0.53
2,germany,1,2,-2.30,-2.30
3,mexico,2,4,0.79,0.40
4,usa,2,6,0.22,0.11
5,portugal,1,1,0.30,0.30
6,switzerland,3,3,1.52,0.51
7,egypt,2,5,-2.61,-1.30
8,argentina,5,11,7.28,1.46
9,norway,3,6,-0.27,-0.09


In [24]:
stage_sheet.head(6)

,Unnamed: 0,stage,logged_matches,incidents,incidents_per_stage,abs_total_match_bias_score,total_match_bias_score,abs_bias_significance_per_match,score_per_match
0,0,R32,14,23,1.642857,13.15,2.02,0.94,0.14
1,1,R16,7,21,3.000000,11.12,1.24,1.59,0.18
2,2,QF,3,6,2.000000,1.99,0.05,0.66,0.02
3,3,SF,2,2,1.000000,1.31,1.31,0.66,0.66
4,4,3P,1,1,1.000000,0.08,-0.08,0.08,-0.08
5,5,F,1,2,2.000000,0.78,0.30,0.78,0.30


In [25]:
incidents.head()

,incident_id,favored_team_hurt,underdog_team_benefited,abs_bias_significance,incident_type,decision,unfairness_bucket,VAR_involved
0,RSA-CAN-R32-01,True,True,1.024909,Potential Penalty,No Penalty,Unfair,False
1,BRA-JPN-R32-01,True,True,0.157744,Foul,No Free Kick,Probably fair,False
2,GER-PAR-R32-01,True,True,0.167705,Potential Penalty,No Penalty,Probably fair,False
3,GER-PAR-R32-02,True,True,2.127630,Foul,Goal Disallowed,robbery,True
4,MEX-ECU-R32-01,False,False,0.096598,Foul,Free Kick,Probably fair,False


Some ranking/matchup previews

In [26]:
#most benefitted teams
df = team_sheet[["team", "impact_per_match"]]
df = df.sort_values("impact_per_match", ascending=False)
df.head(10)

,team,impact_per_match
18,paraguay,2.22
8,argentina,1.46
16,south africa,1.02
26,dr congo,0.94
12,belgium,0.74
1,brazil,0.53
6,switzerland,0.51
3,mexico,0.40
5,portugal,0.30
14,morocco,0.26


In [27]:
#most hurt teams
df = df.sort_values("impact_per_match", ascending=True)
df.head(10)

,team,impact_per_match
24,cape verde,-3.26
2,germany,-2.30
28,senegal,-1.48
7,egypt,-1.30
0,canada,-0.96
22,algeria,-0.88
15,france,-0.68
13,colombia,-0.57
10,england,-0.53
23,australia,-0.33


In [28]:
#matches ranked by total bias calls
df = match_sheet[["match_name", "abs_bias_significance", "bias_direction"]]
df = df.sort_values("abs_bias_significance", ascending=False)
df.head()

,match_name,abs_bias_significance,bias_direction
15,france vs paraguay,3.99,underdog
8,argentina vs cape verde,3.26,favorite
20,argentina vs egypt,2.94,favorite
2,germany vs paraguay,2.30,underdog
4,usa vs bosnia and herzegovina,1.65,neutral


In [29]:
#Mean of abs score when VAR is involved vs not
df = incidents[["VAR_involved", "abs_bias_significance"]]
df = df[df["VAR_involved"] == True]
print("VAR involved: " + df["abs_bias_significance"].mean().round(2).astype(str))
df = incidents[["VAR_involved", "abs_bias_significance"]]
df = df[df["VAR_involved"] == False]
print("VAR not involved: " + df["abs_bias_significance"].mean().round(2).astype(str))


VAR involved: 0.62
VAR not involved: 0.47


In [30]:
#Filter out minor calls, and check again
df = incidents[["VAR_involved", "abs_bias_significance", "unfairness_bucket"]]
df = df[(df["VAR_involved"] == True) & (df["unfairness_bucket"] != "Questionable") & (df["unfairness_bucket"] != "Probably fair")]
print("VAR involved: " + df["abs_bias_significance"].mean().round(2).astype(str))
df = incidents[["VAR_involved", "abs_bias_significance", "unfairness_bucket"]]
df = df[(df["VAR_involved"] == False) & (df["unfairness_bucket"] != "Questionable") & (df["unfairness_bucket"] != "Probably fair")]
print("VAR not involved: " + df["abs_bias_significance"].mean().round(2).astype(str))


VAR involved: 1.09
VAR not involved: 0.85


In [31]:
#Total weighted impact benefitting favs vs unds
df = incidents[["favored_team_hurt", "abs_bias_significance"]]
print(df.groupby([df["favored_team_hurt"]])["abs_bias_significance"].sum().rename(index={
        False: "favorite benefited:",
        True: "underdog benefited:",
    }))


favored_team_hurt
favorite benefited:    16.625711
underdog benefited:    11.788755
Name: abs_bias_significance, dtype: float64


In [32]:
#games where bias = winner
df = match_sheet[["favored_team", "underdog_team", "bias_direction", "upset", "winner"]]
df = df[(df["bias_direction"] == "favorite") & ~(df["upset"]) | (df["bias_direction"] == "underdog")& (df["upset"])]
df = df[["favored_team", "underdog_team", "bias_direction", "winner"]]
df

,favored_team,underdog_team,bias_direction,winner
2,germany,paraguay,underdog,paraguay
6,switzerland,algeria,favorite,switzerland
8,argentina,cape verde,favorite,argentina
12,belgium,senegal,favorite,belgium
14,morocco,canada,favorite,morocco
18,colombia,switzerland,underdog,switzerland
20,argentina,egypt,favorite,argentina
25,spain,france,favorite,spain


In [33]:
#games where bias != winner
df = match_sheet[["favored_team", "underdog_team", "bias_direction", "upset", "winner"]]
df = df[(df["bias_direction"] != "favorite") & ~(df["upset"]) | (df["bias_direction"] != "underdog")& (df["upset"])]
df = df[["favored_team", "underdog_team", "bias_direction", "winner"]]
df

,favored_team,underdog_team,bias_direction,winner
0,canada,south africa,underdog,canada
1,brazil,japan,neutral,brazil
3,mexico,ecuador,neutral,mexico
4,usa,bosnia and herzegovina,neutral,usa
5,portugal,croatia,neutral,portugal
7,egypt,australia,neutral,egypt
9,norway,ivory coast,neutral,norway
10,england,dr congo,underdog,england
11,spain,austria,neutral,spain
13,colombia,ghana,neutral,colombia


In [34]:
#pivot table for teams by calls for/against
#this table is AI generated cause I couldn't figure it out
df = cleaned[["incident_id", "team_benefited", "team_hurt"]].merge(
    incidents[["incident_id", "unfairness_bucket"]],
    on="incident_id",
    how="inner",
)

long = df.melt(
    id_vars=["incident_id", "unfairness_bucket"],
    value_vars=["team_benefited", "team_hurt"],
    var_name="direction",
    value_name="team",
)

long["direction"] = long["direction"].map({
    "team_benefited": "calls for",
    "team_hurt": "calls against",
})

long["bucket_direction"] = (
    long["unfairness_bucket"].str.title() + " " + long["direction"]
)

call_bucket_table = pd.crosstab(long["team"], long["bucket_direction"])

bucket_order = ["probably fair", "questionable", "Minorly unfair", "Unfair", "robbery"]
ordered_cols = [
    f"{bucket.title()} calls {direction}"
    for bucket in bucket_order
    for direction in ["for", "against"]
]

call_bucket_table = (
    call_bucket_table
    .reindex(columns=ordered_cols, fill_value=0)
    .reset_index()
    .sort_values("team")
    .reset_index(drop=True)
)

call_bucket_table

bucket_direction,team,Probably Fair calls for,Probably Fair calls against,Questionable calls for,Questionable calls against,Minorly Unfair calls for,Minorly Unfair calls against,Unfair calls for,Unfair calls against,Robbery calls for,Robbery calls against
0,algeria,0,0,0,0,0,1,0,0,0,0
1,argentina,1,1,4,0,2,0,2,0,1,0
2,australia,0,1,0,1,0,0,0,0,0,0
3,austria,0,0,1,0,0,0,0,0,0,0
4,belgium,1,1,0,0,2,0,0,0,0,0
5,bosnia and herzegovina,0,0,2,1,0,1,0,0,0,0
6,brazil,0,1,1,0,1,0,0,0,0,0
7,canada,0,2,0,0,0,1,0,1,0,0
8,cape verde,0,1,0,1,0,0,0,2,0,0
9,colombia,0,1,0,0,0,1,0,0,0,0
